# 神经网络分布外检测器（ODIN）与 mobilenet

这个 notebook 的目标是理解 [Enhancing The Reliability of Out-of-distribution Image Detection in Neural Networks](https://arxiv.org/abs/1706.02690)。我们把一个训练好的分类器改造成分布外检测器。这里，我们将使用一个在 CIFAR10 上训练的小型 mobilenet 神经网络，看看怎么把它用作检测器，检测通过裁剪 ImageNet 图像得到的分布外样本。

下面的图片取自原论文，展示了 ODIN 方法在 DenseNet-BC-100 网络上的性能。
![](https://raw.githubusercontent.com/ShiyuLiang/odin-pytorch/master/figures/original_optimal_shade.png)

检测器是一个二分类器：当输入图像来自分布内（CIFAR10）时输出 $1$，当输入图像来自分布外（ImageNet 裁剪）时输出 $0$。
CIFAR10 上的 TPR 就是检测器对 CIFAR10 图像判断正确的概率，TinyImagenet（裁剪）上的 FPR 就是检测器对 ImageNet 裁剪图像判断错误的概率。
和普通二分类一样，我们的目标是在最小化 FPR 的同时最大化 TPR，这两个目标之间的权衡由 ROC 曲线刻画（如上图所示）。

在这个实操里，我们用 mobilenet 网络推导 ODIN 方法的 ROC 曲线。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import sys
import os

import matplotlib.pyplot as plt
%matplotlib inline

from sklearn import metrics

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 用于 CIFAR10 的 Mobilenet

我们将使用一个叫 [Mobilenet v2](https://pytorch.org/hub/pytorch_vision_mobilenet_v2/) 的预训练模型，它是一个为速度和内存优化的高效网络，带[残差块](https://dataflowr.github.io/website/modules/17-resnets/)。`torch.vision` 的默认版本是在 Imagenet 上预训练的，所以我们依赖 [PyTorch_CIFAR10](https://github.com/huyvnphan/PyTorch_CIFAR10) 提供的版本，它是在 CIFAR10 上训练的。

下面的代码让你可以下载神经网络的权重，以及对我们有用的[那段代码](https://github.com/dataflowr/notebooks/blob/master/Module17/mobilenetv2.py)（不用克隆上面整个仓库）。


In [ ]:
# 下面的代码下载神经网络的权重，只需要运行一次。
# 你可以取消注释并运行它——第一次使用或你在 colab 上运行时。
#%mkdir state_dicts
#%cd state_dicts/
#!wget https://www.di.ens.fr/~lelarge/mobilenet_v2.pt
#%cd ..

In [ ]:
# 如果在 colab 上，先取消注释并运行下面的命令（只需要一次）：
#!wget https://raw.githubusercontent.com/dataflowr/notebooks/master/Module17/mobilenetv2.py

In [ ]:
from mobilenetv2 import mobilenet_v2

model = mobilenet_v2(pretrained=True)
model.eval()
model.to(DEVICE)

In [ ]:
NORM_BIAS = [125.3 / 255, 123.0 / 255, 113.9 / 255]
NORM_SCALE = [63.0 / 255, 62.1 / 255.0, 66.7 / 255.0]
EPS_FSGM = 1e-2
IMAGE_SIZE = (32, 32)
batch_size = 64

%mkdir data

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(NORM_BIAS, NORM_SCALE),
        transforms.Resize(IMAGE_SIZE),
    ]
)

testset = torchvision.datasets.CIFAR10(root="./data/", train=False, download=True, transform=transform)
testloaderIn = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

## 问题 1：测试你的网络

使用[课程](https://dataflowr.github.io/website/modules/5-stacking-layers/)里的代码（就是我们让 MLP 在 CIFAR10 上过拟合时用的），得到预训练网络在 CIFAR10 测试集上的性能。

数据加载器已经在上面给你了。


In [ ]:
# 你的代码

## 分布外数据集

下面的代码让你下载分布外数据集。
你不应该修改这段代码。


In [ ]:
def imshow(inp, title=None):
#   用于显示 Tensor 的 imshow。
    inp = inp.numpy().transpose((1, 2, 0))
    inp = np.clip(NORM_SCALE * inp + NORM_BIAS, 0,1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)

In [ ]:
batchIn = next(iter(testloaderIn))

In [ ]:
imgIn = batchIn[0]
imgIn = torch.narrow(imgIn, 0, 0, 32)

In [ ]:
im_in = torchvision.utils.make_grid(imgIn)

imshow(im_in,title='images from CIFAR10')

In [ ]:
CIFAR10_labels = ['airplane', 'automobile','bird','cat','deer','dog','frog','horse','ship','truck']
preds = model(imgIn.to(DEVICE)).max(1, keepdim=True)[1]
[CIFAR10_labels[pred] for pred in preds]

In [ ]:
# 下面的代码下载分布外数据集
# 如果在 colab 上，先取消注释并运行下面的命令（只需要一次）：
#%cd data
#!wget https://www.dropbox.com/s/avgm2u562itwpkl/Imagenet.tar.gz
#!tar -xvzf Imagenet.tar.gz
#%cd ..

In [ ]:
testsetout = torchvision.datasets.ImageFolder("./data/Imagenet/", transform=transform)
testloaderOut = DataLoader(testsetout, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
batchOut = next(iter(testloaderOut))

In [ ]:
imgOut = batchOut[0]
imgOut = torch.narrow(imgOut, 0, 0, 32)

In [ ]:
im_out = torchvision.utils.make_grid(imgOut)

imshow(im_out, title='images from ImageNet-crop')

我们现在有两个 dataloader `testloaderIn` 和 `testloaderOut`，分别对应分布内和分布外数据，还有我们的 mobilenet 网络 `model`。

# 分布外检测器

我们的网络 `model` 是在 CIFAR10 上训练的，所以它输出一个大小为 10 的向量，表示属于 CIFAR10 的 10 个类别之一的 log 概率。要把我们的模型用作检测器，需要把这个输出转换成一个单一值，对应"来自分布内"的概率。因此，我们希望这个输出对 CIFAR10 的图像高，对 ImageNet 裁剪的图像低。

## 温度缩放

[ODIN](https://arxiv.org/abs/1706.02690) 提出的第一个方法基于这样的事实：训练好的网络对分布内图像的置信度应该比对分布外图像高。

对图像 ${\bf x}$，设 $f({\bf x}) = (f_1({\bf x}),\dots, f_C({\bf x}))\in \mathbb{R}^C$ 是网络的输出，其中 $C$ 是类别数。对温度缩放参数 $T>0$，我们定义
\begin{eqnarray*}
S_i({\bf x},T) = \frac{\exp(f_i({\bf x})/T)}{\sum_{j=1}^C \exp(f_j({\bf x})/T)}\geq 0,
\end{eqnarray*}
于是 $\sum_i S_i({\bf x},T)=1$。**softmax 分数**就是 $\max_i S_i({\bf x},T)$。

我们预期分布内图像的 softmax 分数高于分布外图像。因此我们可以基于这个分数构建分类器。


## 问题 2：计算 softmax 分数

用 $T=1$ 和 $T=1000$ 计算 softmax 分数，并画出对应的 ROC 曲线（使用 [`metrics.roc_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)）。

提示：要得到和上面论文里类似的图，用 `plt.ylim(0.8, 1.002)`。


In [ ]:
# 这个函数接收一个模型、一个算法和一个 dataloader，计算所有对应的分数
# 要使用这个函数，你的算法需要有一个 apply 方法，它接收
# 一个 batch 和模型作为输入，计算这个 batch 的分数
def compute_scores(model, algo, loader, device=DEVICE):
    model.eval()
    all_scores = []
    for i, (batch, targets) in enumerate(loader):
        bs = batch.shape[0]
        batch = batch.to(device)
        scores = algo.apply(batch, model)
        all_scores += scores
    return all_scores
# 你可以用这样的命令测试你的 apply 方法：
# name_of_your_algo.apply(imgIn.to(DEVICE), model)
# name_of_your_algo.apply(imgOut.to(DEVICE), model)
# 得到上面小 batch 上的分数。

In [ ]:
# 你的代码

## 输入预处理

第二个方法依赖于扰动输入图像。主要想法是如下扰动图像，以提高它的 softmax 分数：
\begin{eqnarray*}
\tilde{\bf x} = {\bf x} - \epsilon \text{sign}\left( -\nabla_{\bf x} \log S_{\hat{y}}({\bf x},T)\right),
\end{eqnarray*}
其中 $\hat{y} =\arg\max_i S_i({\bf x},T)$，$\epsilon>0$ 是施加在图像上的噪声幅度。
这种扰动对 CIFAR10 图像（分布内）的影响应该比对 ImageNet 图像（分布外）大得多，让检测更容易。

第二个方法可以和温度缩放一起用（这就是为什么上面的公式里保留了参数 $T$）。注意，这个方法不需要图像的标签，因此它既可以用于域内图像，也可以用于域外图像。梯度可以用标准的[交叉熵损失](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)计算，但注意梯度是对图像的像素求的，而不是对神经网络的权重（权重保持不动）。类似的技术在[对抗攻击](https://dataflowr.github.io/website/homework/2-CAM-adversarial/)中也用过，那里目标是扰动图像来骗过神经网络做出的预测。


## 问题 3：编写图像扰动，以及带扰动和温度缩放的 ODIN 算法。

提示：
- 克隆图像并让它可微：
```
inputs = images.clone()
inputs.requires_grad=True
```
- 用 `argmax` 计算标签、计算损失并反向传播。
- 用 [`torch.ge`](https://pytorch.org/docs/stable/generated/torch.ge.html#torch-ge) 之类的函数计算梯度的符号，得到二值的梯度
- 记住图像先用 `NORM_SCALE` 缩放过了，所以把二值梯度除以这些量，才能精确实现上面描述的算法（结果也会更好！）。
- 对 ODIN 算法，像上面那样创建一个带 `apply` 方法的类，这样就能复用你计算分数的代码。你可以用这样的命令测试你的算法
```
scores_In_odin = odin.apply(imgIn.to(DEVICE), model)
scores_Out_odin = odin.apply(imgOut.to(DEVICE), model)
```

画出得到的 ROC 曲线。


In [ ]:
# 你的代码